# Project Backend: Microsoft Azure & PostgreSQL Implementation
**Project:** EntiLytics  
**Cloud Provider:** Microsoft Azure for Students

This notebook serves as the formal technical documentation for the EntiLytics database. It covers the initial cloud provisioning via Azure CLI, the relational schema design, the programmatic connection using SQLAlchemy, and known limitations of the current session management implementation.

## Phase 1: Infrastructure Architecture
The following steps were performed using the **Azure CLI** to establish the cloud environment. This ensures the backend is hosted on a persistent, managed service rather than a local machine.

### Provisioning Summary

- **Resource Group:** `entilytics-rg` serves as the logical container for all cloud assets.
- **Deployment Region:** `japanwest`
- **Server SKU:** `Standard_B1ms` (Burstable). Provides cost-efficient performance with burstable CPU credits to handle NLP processing spikes.
- **Network Security:** Connectivity is restricted through defined firewall rules and SSL-encrypted tunnels to ensure secure data transit.

In [ ]:
# The following commands were executed in the terminal to initialize the environment.
# These are kept as a reference for reproducibility.

# Register PostgreSQL Provider
# az provider register --namespace Microsoft.DBforPostgreSQL

# Create Resource Group
# az group create --name entilytics-rg --location japanwest

# Create the Flexible Server
# az postgres flexible-server create \
#     --resource-group entilytics-rg \
#     --name entilytics-db-server \
#     --location japanwest \
#     --admin-user enti_admin \
#     --admin-password <SECURE_PASSWORD> \
#     --sku-name Standard_B1ms \
#     --tier Burstable \
#     --public-access 0.0.0.0

# Create the logical database
# az postgres flexible-server db create \
#     --resource-group entilytics-rg \
#     --server-name entilytics-db-server \
#     --database-name entilytics_db

# Allow Azure-internal services to reach the database
# az postgres flexible-server firewall-rule create \
#     --resource-group entilytics-rg \
#     --name entilytics-db-server \
#     --rule-name allow-azure-services \
#     --start-ip-address 0.0.0.0 \
#     --end-ip-address 0.0.0.0

## Phase 2: Python Environment & Security

Database credentials are not hardcoded into the source code. Instead, a `.env` file is used locally, and **Azure App Service environment variables** are used in production. The `python-dotenv` library loads these credentials at runtime.

### Required Python Packages

The following packages must be present in `requirements.txt` for the database connection to function correctly in the Docker container:

```
sqlalchemy==2.0.48
psycopg2-binary==2.9.9
python-dotenv==1.2.2
```

In [ ]:
import os
import uuid
from datetime import datetime
from dotenv import load_dotenv
from sqlalchemy import create_engine, Column, BigInteger, String, Text, ForeignKey, DateTime
from sqlalchemy.orm import sessionmaker, declarative_base
from sqlalchemy.sql import func

# Load .env from the root directory (local development only)
load_dotenv()

# Retrieve credentials from environment
DB_USER = os.getenv('DB_USER')
DB_PASS = os.getenv('DB_PASS')
DB_HOST = os.getenv('DB_HOST')
DB_NAME = os.getenv('DB_NAME')

# Guard against missing configuration
if not all([DB_USER, DB_PASS, DB_HOST, DB_NAME]):
    raise RuntimeError("Missing database environment variables!")

# Construct the connection string with SSL and a connection timeout
DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:5432/{DB_NAME}?sslmode=require"
engine = create_engine(DATABASE_URL, connect_args={"connect_timeout": 10})

SessionLocal = sessionmaker(bind=engine, autocommit=False, autoflush=False)
Base = declarative_base()

## Phase 3: Relational Schema & ORM Models

The following SQLAlchemy ORM classes map directly to tables in the Azure PostgreSQL database. They allow the application to perform CRUD operations using Python objects instead of raw SQL strings.

### Schema Overview

| Table | Purpose |
|---|---|
| `account` | Stores registered Google OAuth users |
| `user_sessions` | Stores active login sessions with expiry |
| `article` | Stores news article titles and content |
| `summary` | Stores NLP-generated summaries per user |
| `analysis_result` | Stores entity rankings, graph HTML, and JSON |
| `annotation` | Stores user-written notes per article |

In [ ]:
# User Management
class Account(Base):
    __tablename__ = "account"
    accountid = Column(BigInteger, primary_key=True)
    gmail = Column(String(100), unique=True, nullable=False)
    account_role = Column(String(255), default="user")
    created_at = Column(DateTime(timezone=True), server_default=func.now())

class UserSession(Base):
    __tablename__ = "user_sessions"
    session_id = Column(String, primary_key=True, default=lambda: str(uuid.uuid4()))
    gmail = Column(String, ForeignKey("account.gmail"), nullable=False)
    name = Column(String, nullable=False)
    picture = Column(String, nullable=True)
    expires_at = Column(DateTime, nullable=False)
    created_at = Column(DateTime, default=datetime.utcnow)

# Content Storage
class Article(Base):
    __tablename__ = "article"
    articleid = Column(BigInteger, primary_key=True)
    title = Column(Text)
    content = Column(Text)
    created_at = Column(DateTime(timezone=True), server_default=func.now())

# NLP Analysis Results
class Summary(Base):
    __tablename__ = "summary"
    summaryid = Column(BigInteger, primary_key=True)
    accountid = Column(BigInteger, ForeignKey("account.accountid"))
    articleid = Column(BigInteger, ForeignKey("article.articleid"))
    summarytext = Column(Text)

class AnalysisResult(Base):
    """Stores the Relationship Map and Ranked Entities."""
    __tablename__ = "analysis_result"
    resultid = Column(BigInteger, primary_key=True)
    articleid = Column(BigInteger, ForeignKey("article.articleid"))
    entities_all_json = Column(Text)
    rankings_json = Column(Text)
    graph_html = Column(Text)

# User Interactions
class Annotation(Base):
    __tablename__ = "annotation"
    annotationid = Column(BigInteger, primary_key=True)
    accountid = Column(BigInteger, ForeignKey("account.accountid"))
    articleid = Column(BigInteger, ForeignKey("article.articleid"))
    note = Column(Text)

## Phase 4: Session Persistence [Current Limitation]

### How Session Management Works

Upon a successful Google OAuth login, a unique `session_id` (UUID) is generated and stored in:
1. The `user_sessions` table in the Azure PostgreSQL database
2. The user's browser as a cookie named `entil_sid`

On subsequent page loads, a JavaScript bridge reads the cookie and passes the `session_id` back to the Python backend, which queries the database to restore the user's identity.

### Known Issue: Session Lost on Page Refresh

Currently, when a user refreshes the page, they are returned to the login screen despite having a valid session cookie and a corresponding record in the database.

**Root Cause:** Solara uses reactive variables (e.g., `current_user`, `current_session_id`) to hold application state in memory. When the page is refreshed, the server-side Python process re-initializes and all reactive variables reset to their default values, including `current_user`, which resets to `None`.

The `SessionRestorer` component is designed to recover from this by reading the browser cookie and querying the database, but there is a **timing gap**: the UI renders the `LoginScreen` before the JavaScript bridge has finished reading the cookie and before the database lookup completes. This causes a redirect to login.

### Planned Resolution

The session restoration flow needs to enforce a **loading gate**: the application should render a neutral loading state while the session check is in progress, and only decide whether to show the login screen or dashboard after the check resolves. This prevents the premature render of the login view.

> **Status:** Under active development. The `user_sessions` table and cookie infrastructure are fully functional, the issue is isolated to the front-end rendering order within Solara's reactive component lifecycle.

## Phase 5: Database Initialization

The `init_db()` function synchronizes the ORM models with the live database. It is only called directly when running `database.py` as a standalone script, it is **not** called at import time to avoid blocking application startup.

In [ ]:
def init_db():
    """Verify and sync ORM models with the Azure PostgreSQL database."""
    try:
        Base.metadata.create_all(bind=engine)
        print("Database models synced with Azure.")
    except Exception as e:
        print(f"Connection/Migration Error: {e}")

if __name__ == "__main__":
    init_db()

## Phase 6: Role Management

User roles are managed directly through the Azure CLI. The `account_role` column in the `account` table accepts two values: `user` (default) and `admin`.
        
Replace `<PASSWORD>` and `<USER_EMAIL>` before running any command.

In [ ]:
# Check the current role of a specific account

        az postgres flexible-server execute -n entilytics-db-server -u enti_admin -p '<PASSWORD>' -d postgres
            --querytext "SELECT gmail, account_role FROM account WHERE gmail = '<USER_EMAIL>';"

In [ ]:
# Promote User to Admin

        az postgres flexible-server execute -n entilytics-db-server -u enti_admin -p '<your_password>' -d postgres
            --querytext "UPDATE account SET account_role = 'admin' WHERE gmail = 'target-email@gmail.com';"

In [ ]:
# Demote an account back to user
        az postgres flexible-server execute -n entilytics-db-server -u enti_admin -p '<PASSWORD>' -d postgres
            --querytext "UPDATE account SET account_role = 'user' WHERE gmail = '<USER_EMAIL>';"

In [ ]:
# List all accounts and their current roles
        az postgres flexible-server execute -n entilytics-db-server -u enti_admin -p '<PASSWORD>' -d postgres
            --querytext "SELECT gmail, account_role FROM account;"